In [2]:
print('hello world')

hello world


# Group project title

## Group Assignment Template

This notebook serves as the core working template for the group assignment. All required data preparation, modelling, simulation, and visualisation should be completed within this notebook.

## Analytical Workflow

This assignment follows a structured workflow:

1. Parameter Estimation (from Historical Data)
2. Simulation Model Construction
3. Model Testing
4. Monte Carlo Simulation
5. Visualisation and Interpretation
6. What-if Analysis and Decision-Making under Uncertainty

You should build your solution progressively across these steps.
    

## Main Program
### Import Libraries

In [3]:
import platform
import sqlalchemy as sal
import pandas as pd
import random as rnd
from IPython.display import display
from plotnine import *
import numpy as np
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer  #To fill potential missing values
from sklearn.preprocessing import OneHotEncoder #Converts categorical text into numbers that models can understand.
from scipy import stats as scipy_stats # Used for checking Wilson CI for baseline

## Import datas

In [4]:
DATA_path_calander = "calendar_airbnb.csv"          # same folder as this notebook

calandar = pd.read_csv(DATA_path_calander)
print(f"{calandar.shape[0]:,} response, {calandar.shape[1]} columns") #Checking demensions
calandar.head() #Extract first 5 rows

9,390,720 response, 5 columns


,listing_id,date,available,minimum_nights,maximum_nights
0,10803.0,2026-06-17,f,12,28
1,10803.0,2026-06-18,f,12,28
2,10803.0,2026-06-19,t,12,28
3,10803.0,2026-06-20,t,12,28
4,10803.0,2026-06-21,t,12,28


In [5]:
DATA_path_listings = "listings_airbnb.csv"          # same folder as this notebook

listings = pd.read_csv(DATA_path_listings)
print(f"{listings.shape[0]:,} response, {listings.shape[1]} columns") #Checking demensions
listings.head() #Extract first 5 rows

25,728 response, 90 columns


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,10803.0,https://www.airbnb.com/rooms/10803,20260616211523,2026-06-17,city scrape,"Room in Deco Apartment, Brunswick East",A large air conditioned room with firm queen s...,NaN,https://a0.muscache.com/pictures/e5f30dd1-ac57...,38901.0,...,4.73,4.70,4.66,NaN,NaN,1,0,1,0,1.31
1,12936.0,https://www.airbnb.com/rooms/12936,20260616211523,2026-06-28,previous scrape,St Kilda 1BR+BEACHSIDE+BALCONY+WIFI+AC,RIGHT IN THE HEART OF ST KILDA! It doesn't get...,NaN,https://a0.muscache.com/pictures/59701/2e8cdaf...,50121.0,...,4.83,4.78,4.66,NaN,NaN,10,10,0,0,0.22
2,41836.0,https://www.airbnb.com/rooms/41836,20260616211523,2026-06-28,previous scrape,CLOSE TO CITY & MELBOURNE AIRPORT,Easy to travel from and to the Airport; quiet ...,NaN,https://a0.muscache.com/pictures/569696dd-1ad0...,182833.0,...,4.83,4.39,4.69,NaN,NaN,2,0,2,0,0.83
3,43429.0,https://www.airbnb.com/rooms/43429,20260616211523,2026-06-17,city scrape,Tranquil Javanese Studio and Pond!,"No service/Cleaning Fees, EV Charger, Study th...",NaN,https://a0.muscache.com/pictures/airflow/Hosti...,189684.0,...,4.94,4.79,4.86,NaN,NaN,2,2,0,0,1.48
4,44699.0,https://www.airbnb.com/rooms/44699,20260616211523,2026-06-17,city scrape,"15 yearsHosting (4.8), 8 CITY TRAMS, GymPoolTe...",Unwavering service — just ask and we do our be...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,189245.0,...,4.97,4.84,4.71,NaN,NaN,1,0,1,0,0.35


In [6]:
DATA_path_reviews = "reviews_airbnb.csv"          # same folder as this notebook

reviews = pd.read_csv(DATA_path_reviews)
print(f"{reviews.shape[0]:,} response, {reviews.shape[1]} columns") #Checking demensions
reviews.head() #Extract first 5 rows

1,026,690 response, 6 columns


,listing_id,id,date,reviewer_id,reviewer_name,comments
0,10803.0,3333588.0,2013-01-12,4421189.0,Johannes,It was very convenient to stay in Lindsay's a...
1,10803.0,3369053.0,2013-01-18,1763045.0,Julie,Perfect isnt enough! Lindsay was the best host...
2,10803.0,3403930.0,2013-01-23,4423532.0,Ivonne,Living with Lindsay was very relaxed. The room...
3,10803.0,3514479.0,2013-02-11,4551787.0,Jess,"Beautiful home, great location very friendly a..."
4,10803.0,3662039.0,2013-03-01,632036.0,Yvonne,It was great staying at Lindsay's apartment. H...


In [16]:
print(reviews.info())
print("Missing comments:", reviews["comments"].isna().sum())
print("Duplicate review IDs:", reviews["id"].duplicated().sum())
print("Date range:", reviews["date"].min(), "to", reviews["date"].max())

reviews[["comments"]].duplicated()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1026690 entries, 0 to 1026689
Data columns (total 6 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   listing_id     1026690 non-null  float64
 1   id             1026690 non-null  float64
 2   date           1026690 non-null  object 
 3   reviewer_id    1026690 non-null  float64
 4   reviewer_name  1026689 non-null  object 
 5   comments       1026557 non-null  object 
dtypes: float64(3), object(3)
memory usage: 47.0+ MB
None
Missing comments: 133
Duplicate review IDs: 0
Date range: 2010-08-04 to 2026-06-28


0          False
1          False
2          False
3          False
4          False
           ...  
1026685    False
1026686    False
1026687    False
1026688    False
1026689     True
Length: 1026690, dtype: bool

In [ ]:
reviews_clean = reviews.copy()


In [18]:
#Check duplicated reviews
reviews_dup = reviews.copy()
reviews_dup["comments_normalised"] = (
    reviews["comments"]
    .astype("string")
    .fillna("")
    .str.normalize("NFKC")
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)
duplicate_mask = (
    reviews_dup["comments_normalised"].ne("")
    & reviews_dup["comments_normalised"].duplicated(keep=False)
)

duplicate_comments = reviews.loc[duplicate_mask].copy()

print("Rows belonging to repeated-comment groups:",
      len(duplicate_comments))

Rows belonging to repeated-comment groups: 58283


In [20]:
duplicate_summary = (
    duplicate_comments
    .groupby("comments_normalised")
    .agg(
        occurrences=("id", "size"),
        unique_review_ids=("id", "nunique"),
        unique_listings=("listing_id", "nunique"),
        unique_reviewers=("reviewer_id", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        example_comment=("comments", "first")
    )
    .reset_index()
)

duplicate_summary["word_count"] = (
    duplicate_summary["comments_normalised"]
    .str.split()
    .str.len()
)

duplicate_summary = duplicate_summary.sort_values(
    "occurrences",
    ascending=False
)

display(
    duplicate_summary[
        [
            "occurrences",
            "unique_listings",
            "unique_reviewers",
            "word_count",
            "first_date",
            "last_date",
            "example_comment"
        ]
    ].head(30)
)

KeyError: 'comments_normalised'